# Device 2: B-axis Field Sweep vs Temperature

This notebook analyzes and plots H_scan field sweep data across different temperatures.

Features:
- Loads all temperature sweep data files
- Applies field corrections (fixes erroneous measurement values)
- Plots **normalized** lock-in channels (L*/L*_min) vs corrected signed magnetic field H
- Shows error bars from measurement uncertainties
- Displays temperature dependence of magnetoresistance

**Note:** All channels are normalized by dividing by their minimum value (L1/L1_min, L2/L2_min, etc.) to show relative resistance changes.

In [ ]:
# Import required libraries
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()


# Add parent directory to path for imports
sys.path.append('..')

from scripts.Hscan.H_scan_dataloader import H_scan_dataloader
from scripts.Hscan.H_scan_analyzer import analyze_H_scan_data, get_analysis_summary

## Load and Analyze All Temperature Data

In [ ]:
# Specify data directory
data_dir = Path("../../data/device 2/b_scans/H_scans")

# Find all .dat files
dat_files = sorted(data_dir.glob("*.dat"))

print(f"Found {len(dat_files)} data files:")
for f in dat_files:
    print(f"  - {f.name}")

In [ ]:
# Load and analyze all files
loader = H_scan_dataloader()
analyzed_datasets = []
temperatures = []

print("\nLoading and analyzing files...\n")

for file_path in dat_files:
    try:
        # Load raw data
        raw_data = loader.load_file(str(file_path))
        
        # Analyze (correct fields and calculate statistics)
        analyzed_data = analyze_H_scan_data(raw_data)
        
        # Extract temperature from data
        temp = analyzed_data.Tcryo_K[0]
        
        analyzed_datasets.append(analyzed_data)
        temperatures.append(temp)
        
        print(f"Loaded: {file_path.name}")
        print(f"  Temperature: {temp:.1f} K")
        print(f"  Field corrections: {analyzed_data.num_corrections}")
        print(f"  H range: {analyzed_data.H_signed.min():.3f} to {analyzed_data.H_signed.max():.3f} T")
        print()
        
    except Exception as e:
        print(f"Error loading {file_path.name}: {e}")
        print()

print(f"Successfully loaded {len(analyzed_datasets)} datasets")
print(f"Temperature range: {min(temperatures):.1f} K to {max(temperatures):.1f} K")

## Display Analysis Summary for One Dataset

In [ ]:
# Show detailed analysis for first dataset as example
if len(analyzed_datasets) > 0:
    print(get_analysis_summary(analyzed_datasets[0]))

## Plot All Channels vs Temperature (Normalized)

Creates 4 separate plots (one per lock-in channel) showing normalized field sweep data at all temperatures.
Each channel is normalized by dividing by its minimum value (L*/L*_min).

In [ ]:
# Create 2x2 subplot grid for all channels
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Device 2: B-axis Field Sweep vs Temperature (Normalized)', fontsize=16, fontweight='bold')

# Flatten axes for easier iteration
axes = axes.flatten()

# Define channels to plot with their min attribute names
channels = [
    ('L1', 'L1_Ch1', 'L1_Ch2', 'L1_min'),
    ('L2', 'L2_Ch1', 'L2_Ch2', 'L2_min'),
    ('L3', 'L3_Ch1', 'L3_Ch2', 'L3_min'),
    ('L4', 'L4_Ch1', 'L4_Ch2', 'L4_min')
]

# Create colormap for temperature gradient
colors = plt.cm.coolwarm(np.linspace(0, 1, len(analyzed_datasets)))

# Plot each channel
for ch_idx, (ch_label, ch_data, ch_error, ch_min) in enumerate(channels):
    ax = axes[ch_idx]
    
    # Plot all temperatures for this channel
    for idx, (data, temp, color) in enumerate(zip(analyzed_datasets, temperatures, colors)):
        # Get data for this channel
        H = data.H_signed
        signal = getattr(data, ch_data)
        error = getattr(data, ch_error)
        min_val = getattr(data, ch_min)
        
        # Normalize by minimum value
        signal_norm = signal / abs(min_val)
        error_norm = error / abs(min_val)
        
        # Plot with error bars
        ax.errorbar(
            H,
            signal_norm,
            yerr=error_norm,
            fmt='o-',
            markersize=3,
            linewidth=1.5,
            capsize=2,
            label=f'{temp:.0f} K',
            color=color,
            alpha=0.8
        )
    
    # Formatting
    ax.set_xlabel('H (T)', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{ch_label}/{ch_label}_min', fontsize=12, fontweight='bold')
    ax.set_title(f'{ch_label} Channel (Normalized)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', fontsize=8, ncol=2)

# Adjust layout
plt.tight_layout()
plt.show()

## Individual Channel Plots (Larger Format)

Create separate full-size plots for each normalized channel for better visibility.
Each channel is normalized: L*/L*_min

In [ ]:
# Plot each channel in its own figure
for ch_label, ch_data, ch_error, ch_min in channels:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Plot all temperatures
    for idx, (data, temp, color) in enumerate(zip(analyzed_datasets, temperatures, colors)):
        H = data.H_signed
        signal = getattr(data, ch_data)
        error = getattr(data, ch_error)
        min_val = getattr(data, ch_min)
        
        # Normalize by minimum value
        signal_norm = signal / abs(min_val)
        error_norm = error / abs(min_val)
        
        ax.errorbar(
            H,
            signal_norm,
            yerr=error_norm,
            fmt='o-',
            markersize=4,
            linewidth=2,
            capsize=3,
            label=f'{temp:.0f} K',
            color=color,
            alpha=0.8
        )
    
    # Formatting
    ax.set_xlabel('H (T)', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'{ch_label}/{ch_label}_min', fontsize=14, fontweight='bold')
    ax.set_title(f'Device 2: {ch_label} Channel - B-axis Field Sweep (Normalized)', 
                 fontsize=15, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', fontsize=10, ncol=3, title='Temperature')
    
    plt.tight_layout()
    plt.show()

## Summary Table: Field Corrections by Temperature

In [ ]:
# Create summary table
import pandas as pd

summary_data = []
for data, temp in zip(analyzed_datasets, temperatures):
    summary_data.append({
        'Temperature (K)': f"{temp:.1f}",
        'Data Points': len(data.H_signed),
        'Field Corrections': data.num_corrections,
        'H_min (T)': f"{data.H_signed.min():.4f}",
        'H_max (T)': f"{data.H_signed.max():.4f}",
    })

summary_df = pd.DataFrame(summary_data)
print("\nDataset Summary:")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)